# 02 - Bronze to Silver
Tradução de schemas para Português Brasileiro, normalização de status, parse de datas, higienização financeira com conversão para BRL e cálculo de margens, tratamento de column shift, desduplicação de registros, unificação de pessoas e empresas, Forward Fill na cotação do dólar e boas práticas dos quality checks.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Parâmetros de Padronização
catalog = "workspace"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

# Checks de boas práticas para Data Quality
dq_results = []

def dq_check_condition(table_name: str, check_name: str, df, condition_expr: str):
    total = df.count()
    failed = df.filter(~F.expr(condition_expr)).count()
    passed = (failed == 0)
    
    dq_results.append((
        table_name, check_name, total, failed, "PASS" if passed else "FAIL", datetime.now()
    ))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {failed}/{total} falhas")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = (dupes == 0)
    
    dq_results.append((
        table_name, check_name, total, dupes, "PASS" if passed else "FAIL", datetime.now()
    ))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {dupes} chaves duplicadas")

In [0]:
df_info_bronze = spark.table(f"{bronze_schema}.tb_movies_info")

# Normalização de texto e tradução de Status
df_info_temp = df_info_bronze.withColumn(
    "status_clean",
    F.lower(F.trim(F.regexp_replace(F.col("status"), r"[^a-zA-Z\s]", "")))
).withColumn(
    "status_filme",
    F.when(F.col("status_clean").contains("released"), "Lançado")
     .when(F.col("status_clean").contains("post production"), "Pós-Produção")
     .when(F.col("status_clean").contains("in production"), "Em Produção")
     .when(F.col("status_clean").contains("planned"), "Planejado")
     .when(F.col("status_clean").contains("rumored"), "Rumores")
     .when(F.col("status_clean").contains("canceled") | F.col("status_clean").contains("cancelled"), "Cancelado")
     .otherwise("Não Informado")
)

# Parse de data
df_info_temp = df_info_temp.withColumn(
    "data_lancamento_parsed",
    F.coalesce(
        F.try_to_date(F.col("release_date"), F.lit("yyyy-MM-dd")),
        F.try_to_date(F.col("release_date"), F.lit("MM-dd-yyyy")),
        F.try_to_date(F.col("release_date"), F.lit("dd/MM/yyyy")),
        F.try_to_date(F.col("release_date"), F.lit("yyyy/MM/dd"))
    )
).withColumn("ano_lancamento", F.year(F.col("data_lancamento_parsed")))

# Chave de texto limpa com tratamento de exceção para anomalias conhecidas
df_info_temp = (
    df_info_temp
    .withColumn("titulo_tratado",
        # Mapeamento cirúrgico para a bagunça de traduções e pontuações do Kevin Hart
        F.when(F.col("title").rlike("(?i)duro de atuar 2|die hart 2.*|die hart: die harter"), F.lit("Die Hart 2"))
         .when(F.col("title").rlike("(?i)duro de atuar$"), F.lit("Die Hart"))
         .otherwise(F.col("title"))
    )
    # Gera a chave de correspondência usando o título tratado
    .withColumn("match_key", F.lower(F.trim(F.regexp_replace(F.col("titulo_tratado"), r"[^a-zA-Z0-9 ]", ""))))
)

# Título limpo + Ano de Lançamento
# Isso funde clones com IDs diferentes da mesma obra, mas preserva sequências e remakes.
w_info = Window.partitionBy("match_key", "ano_lancamento").orderBy(F.col("ingestion_datetime").desc())

df_info_silver = (
    df_info_temp
    .withColumn("rn", F.row_number().over(w_info))
    .filter("rn = 1")
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.col("title").alias("titulo"),                     # Título oficial preservado intacto
        F.col("original_title").alias("titulo_original"),   # Título original preservado
        F.col("data_lancamento_parsed").alias("data_lancamento"),
        F.col("ano_lancamento").cast("int"),
        F.expr("try_cast(runtime AS INT)").alias("duracao_minutos"), 
        F.col("original_language").alias("idioma_original"),
        F.col("status_filme"),
        F.col("overview").alias("sinopse"),
        F.col("tagline").alias("frase_divulgacao")
    )
)

# DQ Checks
dq_check_unique("tb_info_filmes", "unicidade_id_filme", df_info_silver, ["id_filme"])
dq_check_condition("tb_info_filmes", "id_filme_nao_nulo", df_info_silver, "id_filme IS NOT NULL")

# Persistência na Silver com overwrite de esquema autorizado
df_info_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_info_filmes")

[FAIL] tb_info_filmes | unicidade_id_filme | 3 chaves duplicadas
[PASS] tb_info_filmes | id_filme_nao_nulo | 0/95271 falhas


In [0]:
df_cotacao_bronze = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

# Parsing da data e remoção de duplicados
df_cotacao_base = (
    df_cotacao_bronze
    .withColumn("data_cotacao", F.to_date(F.expr("try_cast(dataHoraCotacao AS timestamp)"))) # <--- CORREÇÃO USANDO F.expr
    .select("data_cotacao", F.col("cotacaoCompra").cast("double").alias("taxa_dolar"))
    .filter("data_cotacao IS NOT NULL")
    .dropDuplicates(["data_cotacao"])
)

# Geração de calendário contínuo
min_max = df_cotacao_base.select(F.min("data_cotacao").alias("min_d"), F.max("data_cotacao").alias("max_d")).collect()[0]

df_calendar = spark.range(0, (min_max["max_d"] - min_max["min_d"]).days + 1).select(
    F.expr(f"date_add(date'{min_max['min_d']}', cast(id as int))").alias("data_dia")
)

# Forward Fill usando Window Function
w_ffill = Window.orderBy("data_dia").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_cotacao_silver = (
    df_calendar
    .join(df_cotacao_base, df_calendar.data_dia == df_cotacao_base.data_cotacao, "left")
    .withColumn("cotacao_dolar", F.last("taxa_dolar", ignorenulls=True).over(w_ffill))
    .select(F.col("data_dia").alias("data_cotacao"), F.col("cotacao_dolar"))
)

# DQ Checks
dq_check_unique("tb_cotacao_dolar", "unicidade_data_cotacao", df_cotacao_silver, ["data_cotacao"])
dq_check_condition("tb_cotacao_dolar", "cotacao_nao_nula", df_cotacao_silver, "cotacao_dolar IS NOT NULL")

df_cotacao_silver.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] tb_cotacao_dolar | unicidade_data_cotacao | 0 chaves duplicadas
[PASS] tb_cotacao_dolar | cotacao_nao_nula | 0/5 falhas


In [0]:
df_fin_bronze = spark.table(f"{bronze_schema}.tb_movies_financials")
df_info_ref = spark.table(f"{silver_schema}.tb_info_filmes").select("id_filme", "data_lancamento")
df_cot_ref = spark.table(f"{silver_schema}.tb_cotacao_dolar")


# Adiciona/Atualiza esta função com a regra do numeric_val <= 0
def clean_numeric_col(col_name):
    cleaned = F.regexp_replace(F.col(col_name), r"[^\d.]", "")
    numeric_val = cleaned.cast("decimal(18,2)")
    return F.when(
        (F.col(col_name).isin("Unknown", "Não Informado", "NULL", "null", "None")) | 
        (cleaned == "") | 
        (numeric_val.isNull()) |
        (numeric_val <= 0),  # <-- Transforma 0, 0.00 e negativos em NULL real
        F.lit(None)
    ).otherwise(numeric_val)

df_fin_clean = (
    df_fin_bronze
    .select(
        F.col("id").cast("string").alias("id_filme"),
        clean_numeric_col("budget").alias("orcamento_usd"),
        clean_numeric_col("revenue").alias("receita_usd")
    )
    .dropDuplicates(["id_filme"])
)

# Cruzamento blindado garantindo que a referência não multiplica linhas
df_info_unique = df_info_ref.dropDuplicates(["id_filme"])

df_fin_joined = (
    df_fin_clean
    .join(df_info_unique, "id_filme", "left")
    .join(df_cot_ref, df_info_ref.data_lancamento == df_cot_ref.data_cotacao, "left")
    .dropDuplicates(["id_filme"])
)

# Cotação fallback (mais recente)
latest_rate = df_cot_ref.orderBy(F.col("data_cotacao").desc()).first()["cotacao_dolar"]

df_fin_silver = (
    df_fin_joined
    .withColumn("taxa_aplicada", F.coalesce(F.col("cotacao_dolar"), F.lit(latest_rate)))
    .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * F.col("taxa_aplicada"), 2))
    .withColumn("receita_brl", F.round(F.col("receita_usd") * F.col("taxa_aplicada"), 2))
    .withColumn("lucro_usd", F.col("receita_usd") - F.col("orcamento_usd"))
    .withColumn("lucro_brl", F.round(F.col("receita_brl") - F.col("orcamento_brl"), 2))
    .withColumn(
        "margem_lucro_pct",
        F.when(
            (F.col("orcamento_usd").isNotNull()) & (F.col("orcamento_usd") > 0),
            F.round((F.col("lucro_usd") / F.col("orcamento_usd")) * 100, 2)
        ).otherwise(F.lit(None))
    )
    .select(
        "id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
        "lucro_usd", "lucro_brl", "margem_lucro_pct"
    )
)

# DQ Checks
dq_check_unique("tb_financeiro_filmes", "unicidade_id_filme", df_fin_silver, ["id_filme"])

# ESCRITA BLINDADA
(
    df_fin_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")
)

[PASS] tb_financeiro_filmes | unicidade_id_filme | 0 chaves duplicadas


In [0]:
df_met_bronze = spark.table(f"{bronze_schema}.tb_movies_metrics")

# Conversão com o try_cast para lidar com o Column Shift
df_met_safe = (
    df_met_bronze
    # Identifica se a string é um ano entre 1800-2099 (4 dígitos) que vazou da coluna vizinha
    .withColumn("is_ano_vazado", F.col("popularity").rlike(r"^(18|19|20)\d{2}$"))
    # Se for ano, anula. Se não, aplica o replace da vírgula e tenta converter para Double
    .withColumn("pop_clean", 
        F.when(F.col("is_ano_vazado"), F.lit(None))
         .otherwise(F.expr("try_cast(regexp_replace(popularity, ',', '.') AS DOUBLE)"))
    )
    .withColumn("vote_avg_clean", F.expr("try_cast(vote_average AS DOUBLE)"))
    .withColumn("vote_cnt_clean", F.expr("try_cast(vote_count AS INT)"))
    .withColumn("rating_clean", F.expr("try_cast(averageRating AS DOUBLE)"))
    .withColumn("num_votes_clean", F.expr("try_cast(numVotes AS INT)"))
)

# Filtro dos limites de negócio na seleção final
df_met_silver = (
    df_met_safe
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.when(F.col("pop_clean") >= 0, F.col("pop_clean")).otherwise(None).alias("popularidade"),
        F.when((F.col("vote_avg_clean") >= 0) & (F.col("vote_avg_clean") <= 10), F.col("vote_avg_clean")).otherwise(None).alias("nota_media_tmdb"),
        F.when(F.col("vote_cnt_clean") >= 0, F.col("vote_cnt_clean")).otherwise(None).alias("qtd_votos_tmdb"),
        F.when((F.col("rating_clean") >= 0) & (F.col("rating_clean") <= 10), F.col("rating_clean")).otherwise(None).alias("nota_media_imdb"),
        F.when(F.col("num_votes_clean") >= 0, F.col("num_votes_clean")).otherwise(None).alias("qtd_votos_imdb")
    )
    .dropDuplicates(["id_filme"])
)

# DQ Checks
dq_check_unique("tb_metricas_engajamento", "unicidade_id_filme", df_met_silver, ["id_filme"])
dq_check_condition("tb_metricas_engajamento", "faixa_nota_tmdb", df_met_silver, "nota_media_tmdb IS NULL OR (nota_media_tmdb >= 0 AND nota_media_tmdb <= 10)")

# Persistência
df_met_silver.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

[PASS] tb_metricas_engajamento | unicidade_id_filme | 0 chaves duplicadas
[PASS] tb_metricas_engajamento | faixa_nota_tmdb | 0/95115 falhas


In [0]:
df_rev_bronze = spark.table(f"{bronze_schema}.tb_movies_reviews")

# Conversão com try_cast para lidar com Column Shift
df_rev_safe = df_rev_bronze.withColumn("nota_clean", F.expr("try_cast(nota AS DOUBLE)"))

df_rev_silver = (
    df_rev_safe
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.trim(F.col("nome")).alias("nome_usuario"),
        F.when((F.col("nota_clean") >= 0) & (F.col("nota_clean") <= 10), F.col("nota_clean")).otherwise(None).alias("nota_usuario"),
        F.when(
            (F.col("comentario").isNull()) | (F.trim(F.col("comentario")) == ""),
            F.lit("Sem comentário")
        ).otherwise(F.trim(F.col("comentario"))).alias("comentario_usuario")
    )
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
)

# DQ Checks
dq_check_condition("tb_avaliacoes_usuarios", "comentario_nao_vazio", df_rev_silver, "comentario_usuario IS NOT NULL AND length(comentario_usuario) > 0")

# Persistência
df_rev_silver.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

[PASS] tb_avaliacoes_usuarios | comentario_nao_vazio | 0/32412 falhas


In [0]:
# Lista de géneros válidos oficiais do TMDB para blindar contra lixo textual
GENEROS_VALIDOS = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
    "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery",
    "Romance", "Science Fiction", "Thriller", "TV Movie", "War", "Western"
]

# Lê a tabela bronze necessária para os géneros
df_cred_bronze = spark.table(f"{bronze_schema}.tb_credits_and_tags")

# Padroniza separador e faz o explode na coluna correta (genres)
df_gen_exploded = (
    df_cred_bronze
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.explode(F.split(F.regexp_replace(F.col("genres"), r"[;,]", ","), ",")).alias("nome_genero_raw")
    )
)

# Limpeza, padronização e filtro estricto pelos géneros válidos
df_gen_silver = (
    df_gen_exploded
    .withColumn("nome_genero", F.initcap(F.trim(F.col("nome_genero_raw"))))
    .filter(F.col("nome_genero").isin(GENEROS_VALIDOS))
    .select("id_filme", "nome_genero")
    .dropDuplicates(["id_filme", "nome_genero"])
)

# DQ Checks
dq_check_unique("tb_generos", "unicidade_filme_genero", df_gen_silver, ["id_filme", "nome_genero"])

# Persistência
df_gen_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_generos")

[PASS] tb_generos | unicidade_filme_genero | 0 chaves duplicadas


In [0]:
df_cred_bronze = spark.table(f"{bronze_schema}.tb_credits_and_tags")

def process_entity_col(col_name, entity_type):
    return (
        df_cred_bronze
        .select(
            F.col("id").cast("string").alias("id_filme"),
            F.explode(F.split(F.regexp_replace(F.col(col_name), r";", ","), r",")).alias("nome_raw")
        )
        # Normalização rigorosa: remove espaços duplicados internos, trim e initcap
        .withColumn("nome_entidade", F.initcap(F.trim(F.regexp_replace(F.col("nome_raw"), r"\s+", " "))))
        .withColumn("tipo_entidade", F.lit(entity_type))
        .filter(
            "nome_entidade IS NOT NULL AND length(nome_entidade) > 0 AND nome_entidade NOT RLIKE '^[0-9]+$'"
        )
        .select("id_filme", "nome_entidade", "tipo_entidade")
    )

df_atores = process_entity_col("cast", "Ator")
df_diretores = process_entity_col("directors", "Diretor")
df_roteiristas = process_entity_col("writers", "Roteirista")
df_produtoras = process_entity_col("production_companies", "Produtora")

df_pessoas_empresas_silver = (
    df_atores
    .union(df_diretores)
    .union(df_roteiristas)
    .union(df_produtoras)
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

# DQ Checks
dq_check_unique("tb_pessoas_empresas", "unicidade_entidade_filme", df_pessoas_empresas_silver, ["id_filme", "nome_entidade", "tipo_entidade"])

# Persistência com overwriteSchema
df_pessoas_empresas_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

[PASS] tb_pessoas_empresas | unicidade_entidade_filme | 0 chaves duplicadas


In [0]:
# Consolida a tabela de auditoria de qualidade de dados na camada Silver
schema_dq = StructType([
    StructField("tabela", StringType(), True),
    StructField("checagem", StringType(), True),
    StructField("total_linhas", IntegerType(), True),
    StructField("linhas_falhas", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("checked_at", TimestampType(), True)
])

df_dq_log = spark.createDataFrame(dq_results, schema=schema_dq)
df_dq_log.write.format("delta").mode("append").saveAsTable(f"{silver_schema}.tb_data_quality_logs")

display(df_dq_log)

tabela,checagem,total_linhas,linhas_falhas,status,checked_at
tb_info_filmes,unicidade_id_filme,95271,3,FAIL,2026-09-21T14:52:12.547Z
tb_info_filmes,id_filme_nao_nulo,95271,0,PASS,2026-09-21T14:52:16.515Z
tb_cotacao_dolar,unicidade_data_cotacao,5,0,PASS,2026-09-21T14:52:24.200Z
tb_cotacao_dolar,cotacao_nao_nula,5,0,PASS,2026-09-21T14:52:25.664Z
tb_financeiro_filmes,unicidade_id_filme,99006,0,PASS,2026-09-21T14:52:31.283Z
tb_metricas_engajamento,unicidade_id_filme,95115,0,PASS,2026-09-21T14:52:37.717Z
tb_metricas_engajamento,faixa_nota_tmdb,95115,0,PASS,2026-09-21T14:52:39.206Z
tb_avaliacoes_usuarios,comentario_nao_vazio,32412,0,PASS,2026-09-21T14:52:45.100Z
tb_generos,unicidade_filme_genero,129100,0,PASS,2026-09-21T14:52:49.491Z
tb_pessoas_empresas,unicidade_entidade_filme,915151,0,PASS,2026-09-21T14:53:03.725Z


In [0]:
# Manutenção e Otimização das Tabelas Delta
# O OPTIMIZE compacta arquivos pequenos para melhorar a leitura
# O ZORDER cria um índice multidimensional na coluna id_filme

tabelas_silver = [
    "tb_info_filmes", "tb_financeiro_filmes", "tb_metricas_engajamento", 
    "tb_avaliacoes_usuarios", "tb_generos", "tb_pessoas_empresas"
]

for tabela in tabelas_silver:
    print(f"Otimizando {tabela}...")
    
    # Se a tabela tiver id_filme, a gente faz o Z-Order por ela para acelerar o downstream (Star Schema)
    try:
        spark.sql(f"OPTIMIZE {silver_schema}.{tabela} ZORDER BY (id_filme)")
    except:
        spark.sql(f"OPTIMIZE {silver_schema}.{tabela}") # Fallback caso a tabela não tenha id_filme
        
print("Manutenção concluída. Camada Silver finalizada e otimizada com sucesso.")

Otimizando tb_info_filmes...
Otimizando tb_financeiro_filmes...
Otimizando tb_metricas_engajamento...
Otimizando tb_avaliacoes_usuarios...
Otimizando tb_generos...
Otimizando tb_pessoas_empresas...
Manutenção concluída. Camada Silver finalizada e otimizada com sucesso.


In [0]:
# Validação das Tabelas Silver
tabelas_para_validar = [
    "tb_info_filmes", 
    "tb_financeiro_filmes", 
    "tb_metricas_engajamento", 
    "tb_avaliacoes_usuarios", 
    "tb_generos", 
    "tb_pessoas_empresas",
    "tb_cotacao_dolar"
]

for tabela in tabelas_para_validar:
    print(f"\nAmostra de 10 registros da tabela: {silver_schema}.{tabela}")
    display(spark.table(f"{silver_schema}.{tabela}").limit(10))


Amostra de 10 registros da tabela: workspace.silver.tb_info_filmes


id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao
482997,★,★,2017-10-23,2017,98,xx,Lançado,The stars in the night's sky. Pinpricks of light against the darkness excerpted from films beginning at cinema's dawn and continuing to this present day in a project that is planned to be expanded yearly.,The ultimate voyage through space and time.
584007,"Однажды в Америке, или Чисто русская сказка","Однажды в Америке, или Чисто русская сказка",2019-01-01,2019,0,ru,Lançado,null,null
839649,Подслушано,Подслушано,2021-05-31,2021,0,ru,Lançado,null,null
1067037,叫我郑先生,叫我郑先生,2022-11-11,2022,0,zh,Lançado,null,null
1121144,0009: The Sharks Make Contact,0009: The Sharks Make Contact,2020-09-16,2020,133,en,Lançado,"In the sharks' second adventure, a shark's refusal to return a glass of pesto leads to a terrorist crisis.",Experience a movie featuring motion in color
929813,0.1% World,好想去你的世界爱你,2022-02-14,2022,0,zh,Lançado,\Tells the love story between men and women in different places who perceive each other's emotions and perceptions and communicate. Because of an accident,assistant architect An Yi and piano tuner Gao Ang become connected through brain waves. Since then
536024,02-06,洞兩洞六,2021-02-19,2021,23,zh,Lançado,"Made with fake blood and anger, 02-06 tells a horror story about being on sentry duty forever. If you have served in the army, you know how terrifying it is. If you have not, being on sentry duty actually isn't as hard as your job.",null
1170383,096,096,2020-03-11,2020,24,en,Lançado,"Once SCP-096 is re-contained after breaching containment, fleeing the facility it was contained in and killing 63 individuals in a genocidal rage, Inspector Arlia interviews Doctor Daniels, the man responsible for the containment of SCP-096. They discuss the terrible and gruesome events that carried out during the 096-1-A incident. Doctor Daniels is given authority from the 05 Council to begin termination methods on SCP-096, in the most humane way possible.",null
586032,1,1,2020-09-14,2020,100,en,Lançado,"Early morning silence is broken by screeching tires as a helicopter bears down on a speeding vehicle. Taking a quick corner, the team tumbles out into the woods as their car pulls away. Now they must make their way through the thick of nature and thick gunfire to accomplish their mission. Not a single word of dialogue is spoken throughout the entire film. Instead, the music, sounds, images and deeply truthful acting turn a simple plot into an intense experience. Passion and intrigue keep building to the very end.",null
499819,1 1/2 Hours,1 1/2 Hora,2017-05-01,2017,82,es,Lançado,"Long take uncut about ANA's life, who is recently divorced and has a daughter, ANA has started a relationship but she want anybody knows about it. That morning Ana received a call saying her daughter is call and she must to pick her at school, but at school no one knows where his daughter is. There she received a unknown call from the kidnapper who has her daughter and they begin to blackmail her. If she wants her daughter back, she must to do anything and gradually she become isolated. Finally ANA is face to face with the kidnapper.",null



Amostra de 10 registros da tabela: workspace.silver.tb_financeiro_filmes


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct
259316,180000000.00,809342332.00,9.28242E8,4.17369747189E9,629342332.00,3.24545547189E9,349.63
324786,40000000.00,175302354.00,2.06276E8,9.0401670934E8,135302354.00,6.9774070934E8,338.26
339846,69000000.00,177856751.00,3.558261E8,9.1718947923E8,108856751.00,5.6136337923E8,157.76
353081,178000000.00,791657398.00,9.179282E8,4.08249803575E9,613657398.00,3.16456983575E9,344.75
406997,20000000.00,305937718.00,1.03138E8,1.57769021795E9,285937718.00,1.47455221795E9,1429.69
300669,9900000.00,159047649.00,5.105331E7,8.2019282113E8,149147649.00,7.6913951113E8,1506.54
376867,4000000.00,65046687.00,2.06276E7,3.3543926019E8,61046687.00,3.1481166019E8,1526.17
479455,110000000.00,253890701.00,5.67259E8,1.30928895599E9,143890701.00,7.4202995599E8,130.81
640146,200000000.00,476071180.00,1.03138E9,2.45505146814E9,276071180.00,1.42367146814E9,138.04
301337,68000000.00,55003890.00,3.506692E8,2.8364956034E8,-12996110.00,-6.701963966E7,-19.11



Amostra de 10 registros da tabela: workspace.silver.tb_metricas_engajamento


...

id_filme,nome_genero
353486,Comedy
337167,Romance
613504,Drama
301365,Horror
503314,Science Fiction
531454,Comedy
374617,Mystery
502425,Drama
502425,History
401898,Thriller



Amostra de 10 registros da tabela: workspace.silver.tb_pessoas_empresas


id_filme,nome_entidade,tipo_entidade
398978,Ray Romano,Ator
399057,Colin Farrell,Ator
397837,Cynthy Wu,Ator
400650,Ben Whishaw,Ator
14564,Patrick Walker,Ator
267193,Isla Fisher,Ator
400579,Elaine Tan,Ator
675445,Tyler Perry,Ator
419831,Jennifer Ehle,Ator
426613,Marin Ireland,Ator



Amostra de 10 registros da tabela: workspace.silver.tb_cotacao_dolar


data_cotacao,cotacao_dolar
2026-09-14,5.169
2026-09-15,5.1484
2026-09-16,5.152
2026-09-17,5.1515
2026-09-18,5.1569
